## 5. Modelado y evaluación

In [12]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from tensorflow.keras import layers, Model

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

import optuna


# Paths
X_train_in = Path("../data/modeling/supervised/X_train.parquet")
X_test_in = Path("../data/modeling/supervised/X_test.parquet")
y_train_in = Path("../data/modeling/supervised/y_train.parquet")
y_test_in = Path("../data/modeling/supervised/y_test.parquet")

X_train_N_in = Path("../data/modeling/unsupervised/X_train_N.parquet")
X_test_N_in = Path("../data/modeling/unsupervised/X_test_N.parquet")
y_test_N_in = Path("../data/modeling/unsupervised/y_test_N.parquet")

#### 1) Revisión general

Cargo los diferentes conjuntos preparados para usar en los diferentes modelos.

In [14]:
X_train = pd.read_parquet(X_train_in)
X_test = pd.read_parquet(X_test_in)
y_train = pd.read_parquet(y_train_in)
y_test = pd.read_parquet(y_test_in)

X_train_N = pd.read_parquet(X_train_N_in)
X_test_N = pd.read_parquet(X_test_N_in)
y_test_N = pd.read_parquet(y_test_N_in)

Soluciono un par de errores de tipo pd a np: 

In [15]:
y_train = y_train.squeeze().to_numpy()

#### 2) Modelos Supervisados

Dada la separación hecha en el notebook 4 más la codificación de la variable respuesta en este caso '0' representa Ataque y '1' Benigno.

**XGBoost**

In [16]:
# Weights adjustment
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 0.23932138095597458


In [19]:
# Creation Base model of XGBoost
base_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=8,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=5,
    gamma=0.1,

    reg_alpha=0.1,
    reg_lambda=1.0,

    scale_pos_weight=scale_pos_weight,

    tree_method="hist",
    device="cuda",

    eval_metric="logloss",

    random_state=42,
    n_jobs=-1
)

# Train Base model
base_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

[0]	validation_0-logloss:0.66959
[50]	validation_0-logloss:0.21858
[100]	validation_0-logloss:0.14160
[150]	validation_0-logloss:0.12345
[200]	validation_0-logloss:0.11829
[250]	validation_0-logloss:0.11544
[300]	validation_0-logloss:0.11308
[350]	validation_0-logloss:0.11141
[399]	validation_0-logloss:0.11013

=== CONFUSION MATRIX ===
[[ 75263   7982]
 [  6002 341838]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.93      0.90      0.91     83245
           1       0.98      0.98      0.98    347840

    accuracy                           0.97    431085
   macro avg       0.95      0.94      0.95    431085
weighted avg       0.97      0.97      0.97    431085

ROC-AUC: 0.9881636928888873
PR-AUC: 0.9968680807405996


In [ ]:
# PREDICTIONS
y_pred = base_model.predict(X_test)
y_proba = base_model.predict_proba(X_test)[:, 1]

# BASE MODEL EVALUATION
print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))

In [ ]:
# FEATURE IMPORTANCE
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": base_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTop 20 Features:")
print(importance.head(20))

**Random Forest**

In [20]:
# Random Forest uses class_weight instead of scale_pos_weight
rf_model = RandomForestClassifier(
    n_estimators=100,          # increase for final model (500–1000)
    max_depth=10,              # controls overfitting

    min_samples_split=10,
    min_samples_leaf=5,

    max_features="sqrt",

    class_weight="balanced",  # IMPORTANT for IDS imbalance

    n_jobs=-1,
    random_state=42,
    verbose=1
)

# Train model
rf_model.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  2.6min finished


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

In [21]:
# PREDICTIONS
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

# EVALUATION
print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished



=== CONFUSION MATRIX ===
[[ 72704  10541]
 [  1201 346639]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.98      0.87      0.93     83245
           1       0.97      1.00      0.98    347840

    accuracy                           0.97    431085
   macro avg       0.98      0.93      0.95    431085
weighted avg       0.97      0.97      0.97    431085


ROC-AUC: 0.9823301620888797
PR-AUC: 0.9951571257745379


In [ ]:
# FEATURE IMPORTANCE
importances = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTOP 20 FEATURES:")
print(importances.head(20))

#### 3) Modelos No Supervisados

**Isolation Forest**

In [20]:

model = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

#Train Model
model.fit(X_train_scaled_N)

# ANOMALY SCORES
# higher = more anomalous
scores = -model.decision_function(X_test_scaled_N)

# PR CURVE
# convert labels so:
# 1 = attack (positive class)
y_true = 1 - y_test
precision, recall, thresholds = precision_recall_curve(y_true, scores)

# PR-AUC
pr_auc = average_precision_score(y_true, scores)

# PLOT PR CURVE
plt.figure()
plt.plot(recall, precision, label=f"PR AUC = {pr_auc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Isolation Forest Precision-Recall Curve")
plt.legend()
plt.show()

# BEST THRESHOLD (F1 OPTIMIZATION)
# thresholds array is shorter by 1 than precision/recall
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print("\nBest threshold (PR-based):", best_threshold)

# PREDICTIONS
y_pred = (scores >= best_threshold).astype(int)
# convert back:
# 0 = attack, 1 = benign
y_pred = np.where(y_pred == 1, 0, 1)

# EVALUATION
print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_true, scores))
print("PR-AUC:", pr_auc)


Best threshold: -0.1320082152351147

=== CONFUSION MATRIX ===
[[384137  32089]
 [146848 200991]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

         0.0       0.72      0.92      0.81    416226
         1.0       0.86      0.58      0.69    347839

    accuracy                           0.77    764065
   macro avg       0.79      0.75      0.75    764065
weighted avg       0.79      0.77      0.76    764065


ROC-AUC: 0.7019460225811811
PR-AUC: 0.6486871624074865


**Autoencoder**

In [41]:
y_true = 1 - y_test

# =====================================================
# MODEL
# =====================================================
inp = layers.Input(shape=(X_train_scaled_N.shape[1],))

x = layers.Dense(128, activation="relu")(inp)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)

x = layers.Dense(64, activation="relu")(x)
x = layers.Dense(32, activation="relu")(x)

latent = layers.Dense(16, activation="relu")(x)

x = layers.Dense(32, activation="relu")(latent)
x = layers.Dense(64, activation="relu")(x)

x = layers.BatchNormalization()(x)
out = layers.Dense(X_train_scaled_N.shape[1])(x)

autoencoder = Model(inp, out)

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mae"
)

# =====================================================
# TRAIN (benign only)
# =====================================================
autoencoder.fit(
    X_train_scaled_N,
    X_train_scaled_N,
    epochs=40,
    batch_size=128,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)

# =====================================================
# SCORES (reconstruction error)
# =====================================================
recon = autoencoder.predict(X_test_scaled_N, verbose=0)

scores = np.mean(np.abs(X_test_scaled_N - recon), axis=1)

# =====================================================
# THRESHOLD (PR OPTIMIZED)
# =====================================================
precision, recall, thresholds = precision_recall_curve(y_true, scores)
pr_auc = average_precision_score(y_true, scores)

f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-9)
best_threshold = thresholds[np.argmax(f1)]

print("\nBest threshold:", best_threshold)

# =====================================================
# PREDICTIONS
# =====================================================
y_pred = (scores >= best_threshold).astype(int)
y_pred = np.where(y_pred == 1, 0, 1)  # back to: 0 attack, 1 benign

# =====================================================
# EVALUATION
# =====================================================
print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_true, scores))
print("PR-AUC:", pr_auc)

Epoch 1/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 35s 3ms/step - loss: 921.3083 - val_loss: 247.9598
Epoch 2/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 887.6609 - val_loss: 184.6874
Epoch 3/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step - loss: 829.7930 - val_loss: 144.6465
Epoch 4/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 778.5973 - val_loss: 56.7561
Epoch 5/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step - loss: 762.3800 - val_loss: 21.0405
Epoch 6/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 28s 3ms/step - loss: 759.2056 - val_loss: 28.3306
Epoch 7/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - loss: 759.4927 - val_loss: 24.5148
Epoch 8/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 758.5329 - val_loss: 22.9350
Epoch 9/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - loss: 758.7504 - val_loss: 36.9798
Epoch 10/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - loss: 758.6552 - val_loss: 37.6445
Epoch 11/40
9783/9783 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step - loss: 757.8105 - v